# CNN for Age Group Detection

This notebook trains and evaluates a **coursework-ready CNN** for 4-class age-group classification: Child, Young, Middle-Aged, Senior.

It is designed to work in Google Colab or locally, and includes:
- dataset loading from a folder structure with `train/` and `test/`
- automatic train/validation split from the provided training set
- grayscale preprocessing and resizing
- a custom PyTorch CNN trained from scratch
- class-weighted loss for imbalance handling
- early stopping and best-model checkpointing
- quantitative evaluation, confusion matrix, per-class metrics, and qualitative examples

References used in code comments are included where relevant, which is required by the coursework brief [file:1].

## Expected dataset structure

This notebook assumes a dataset root with this structure:

```text
DATASET_ROOT/
├── train/
│   ├── 0/
│   ├── 1/
│   ├── 2/
│   └── 3/
└── test/
    ├── 0/
    ├── 1/
    ├── 2/
    └── 3/
```

If your dataset is organised differently, only the path-loading cell needs small changes. The coursework states the labels are:
- `0 = Child`
- `1 = Young`
- `2 = Middle-Aged`
- `3 = Senior` [file:1]

In [ ]:
# Optional Colab setup
# Reference: adapted from the module lab style for Colab/Drive usage [file:9]

IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
except Exception:
    IN_COLAB = False

print('Running in Colab:', IN_COLAB)

In [ ]:
# Mount Google Drive if using Colab
# Uncomment and run this cell if your dataset/models are on Drive.

# if IN_COLAB:
#     from google.colab import drive
#     drive.mount('/content/drive')

In [ ]:
import os
import random
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    precision_recall_fscore_support,
)
from sklearn.utils.class_weight import compute_class_weight

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Using device:', device)

In [ ]:
# =========================
# USER SETTINGS
# =========================

# Change this to your dataset root.
# Examples:
# DATASET_ROOT = '/content/CWDataset'
# DATASET_ROOT = '/content/drive/MyDrive/CWFolder/CWDataset'
# DATASET_ROOT = './CWDataset'
DATASET_ROOT = '/content/CWDataset'

OUTPUT_DIR = './cnn_outputs'
MODEL_PATH = os.path.join(OUTPUT_DIR, 'best_cnn_age_group.pth')

IMG_SIZE = 128
BATCH_SIZE = 64
NUM_EPOCHS = 20
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 5
NUM_WORKERS = 2

CLASS_NAMES = ['Child', 'Young', 'Middle-Aged', 'Senior']
NUM_CLASSES = len(CLASS_NAMES)

os.makedirs(OUTPUT_DIR, exist_ok=True)
print('Output directory:', OUTPUT_DIR)

In [ ]:
# Dataset utilities
# Reference: custom Dataset pattern adapted from PyTorch docs and the module's CNN lab structure [file:9]

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.tif', '.tiff'}


def find_class_folders(split_dir):
    split_dir = Path(split_dir)
    class_dirs = []
    for class_idx in range(NUM_CLASSES):
        class_path = split_dir / str(class_idx)
        if not class_path.exists():
            raise FileNotFoundError(f'Missing class folder: {class_path}')
        class_dirs.append(class_path)
    return class_dirs


def collect_samples(split_dir):
    split_dir = Path(split_dir)
    class_dirs = find_class_folders(split_dir)
    samples = []
    for class_idx, class_dir in enumerate(class_dirs):
        for file_path in sorted(class_dir.iterdir()):
            if file_path.suffix.lower() in IMG_EXTS:
                samples.append((str(file_path), class_idx))
    if len(samples) == 0:
        raise RuntimeError(f'No image files found in {split_dir}')
    return samples


class FaceAgeDataset(Dataset):
    def __init__(self, samples, transform=None):
        self.samples = samples
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert('L')
        if self.transform is not None:
            image = self.transform(image)
        return image, label

In [ ]:
# Load dataset file paths

train_dir = os.path.join(DATASET_ROOT, 'train')
test_dir = os.path.join(DATASET_ROOT, 'test')

full_train_samples = collect_samples(train_dir)
test_samples = collect_samples(test_dir)

train_labels_full = np.array([label for _, label in full_train_samples])
test_labels = np.array([label for _, label in test_samples])

train_samples, val_samples = train_test_split(
    full_train_samples,
    test_size=0.2,
    random_state=SEED,
    stratify=train_labels_full,
)

print('Full training images:', len(full_train_samples))
print('Training split:', len(train_samples))
print('Validation split:', len(val_samples))
print('Test images:', len(test_samples))

In [ ]:
# Check class balance

def count_by_class(samples):
    labels = [label for _, label in samples]
    counts = {name: labels.count(i) for i, name in enumerate(CLASS_NAMES)}
    return counts

print('Train class counts:', count_by_class(train_samples))
print('Val class counts:  ', count_by_class(val_samples))
print('Test class counts: ', count_by_class(test_samples))

In [ ]:
# Preprocessing and augmentation
# The coursework notes that image resolution varies and preprocessing should account for this [file:1].

train_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=8),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

val_test_transform = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5], std=[0.5]),
])

train_dataset = FaceAgeDataset(train_samples, transform=train_transform)
val_dataset = FaceAgeDataset(val_samples, transform=val_test_transform)
test_dataset = FaceAgeDataset(test_samples, transform=val_test_transform)

pin_memory = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=pin_memory,
)

print('DataLoaders ready.')

In [ ]:
# Visualise a few training examples

def denormalise(img_tensor):
    img = img_tensor * 0.5 + 0.5
    return img.clamp(0, 1)

images, labels = next(iter(train_loader))
fig, axes = plt.subplots(2, 4, figsize=(10, 5))
for ax, img, label in zip(axes.flat, images[:8], labels[:8]):
    ax.imshow(denormalise(img).squeeze(0), cmap='gray')
    ax.set_title(CLASS_NAMES[label.item()])
    ax.axis('off')
plt.suptitle('Example training images')
plt.tight_layout()
plt.show()

In [ ]:
# CNN architecture
# Reference: custom CNN built using standard PyTorch layers following the kind of workflow encouraged in the coursework and CNN lab [file:1][file:9]

class AgeGroupCNN(nn.Module):
    def __init__(self, num_classes=4):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),

            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
        )

        with torch.no_grad():
            dummy = torch.zeros(1, 1, IMG_SIZE, IMG_SIZE)
            feat_dim = self.features(dummy).view(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes),
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

model = AgeGroupCNN(num_classes=NUM_CLASSES).to(device)
print(model)

In [ ]:
# Loss, optimizer, and class weights
# The coursework notes that class imbalance may be present and should be considered [file:1].

train_targets = np.array([label for _, label in train_samples])
class_weights = compute_class_weight(
    class_weight='balanced',
    classes=np.arange(NUM_CLASSES),
    y=train_targets,
)
class_weights_tensor = torch.tensor(class_weights, dtype=torch.float32, device=device)

criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
optimizer = optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=0.5, patience=2)

print('Class weights:', class_weights)

In [ ]:
# Training and validation helpers

def run_one_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    running_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        if is_train:
            optimizer.zero_grad()

        with torch.set_grad_enabled(is_train):
            outputs = model(images)
            loss = criterion(outputs, labels)
            preds = outputs.argmax(dim=1)

            if is_train:
                loss.backward()
                optimizer.step()

        running_loss += loss.item() * images.size(0)
        all_preds.extend(preds.detach().cpu().numpy())
        all_labels.extend(labels.detach().cpu().numpy())

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = accuracy_score(all_labels, all_preds)
    return epoch_loss, epoch_acc, np.array(all_labels), np.array(all_preds)

In [ ]:
# Train the CNN with early stopping

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

best_val_acc = 0.0
best_epoch = -1
epochs_without_improvement = 0
train_start_time = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_start = time.time()

    train_loss, train_acc, _, _ = run_one_epoch(model, train_loader, criterion, optimizer=optimizer)
    val_loss, val_acc, _, _ = run_one_epoch(model, val_loader, criterion, optimizer=None)

    scheduler.step(val_acc)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    epoch_time = time.time() - epoch_start
    current_lr = optimizer.param_groups[0]['lr']
    print(f'Epoch {epoch+1:02d}/{NUM_EPOCHS} | '
          f'train_loss={train_loss:.4f} train_acc={train_acc:.4f} | '
          f'val_loss={val_loss:.4f} val_acc={val_acc:.4f} | '
          f'lr={current_lr:.6f} | time={epoch_time:.1f}s')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_epoch = epoch + 1
        epochs_without_improvement = 0
        torch.save(model.state_dict(), MODEL_PATH)
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print(f'Early stopping triggered after {epoch+1} epochs.')
        break

training_time = time.time() - train_start_time
print(f'Best validation accuracy: {best_val_acc:.4f} at epoch {best_epoch}')
print(f'Total training time: {training_time:.2f} seconds')
print(f'Best model saved to: {MODEL_PATH}')

In [ ]:
# Plot learning curves

epochs = range(1, len(history['train_loss']) + 1)

plt.figure(figsize=(12, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], marker='o', label='Train loss')
plt.plot(epochs, history['val_loss'], marker='o', label='Val loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss curves')
plt.legend()
plt.grid(True)

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_acc'], marker='o', label='Train accuracy')
plt.plot(epochs, history['val_acc'], marker='o', label='Val accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy curves')
plt.legend()
plt.grid(True)

plt.tight_layout()
plt.show()

In [ ]:
# Load best model before final evaluation

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()
print('Best checkpoint loaded.')

In [ ]:
# Evaluation helper

def evaluate_loader(model, loader, split_name='set'):
    loss, acc, y_true, y_pred = run_one_epoch(model, loader, criterion, optimizer=None)
    print(f'\n{split_name} loss: {loss:.4f}')
    print(f'{split_name} accuracy: {acc:.4f}')
    print(f'{split_name} classification report:')
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4))
    return loss, acc, y_true, y_pred

train_loss_final, train_acc_final, y_train_true, y_train_pred = evaluate_loader(model, train_loader, 'Train')
val_loss_final, val_acc_final, y_val_true, y_val_pred = evaluate_loader(model, val_loader, 'Validation')
test_loss_final, test_acc_final, y_test_true, y_test_pred = evaluate_loader(model, test_loader, 'Test')

In [ ]:
# Confusion matrix on the test set

cm = confusion_matrix(y_test_true, y_test_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=CLASS_NAMES)
fig, ax = plt.subplots(figsize=(7, 6))
disp.plot(ax=ax, cmap='Blues', xticks_rotation=30, colorbar=False)
plt.title('CNN confusion matrix on test set')
plt.tight_layout()
plt.show()

In [ ]:
# Per-class precision, recall, and F1

precision, recall, f1, _ = precision_recall_fscore_support(
    y_test_true,
    y_test_pred,
    labels=np.arange(NUM_CLASSES),
    zero_division=0,
)

x = np.arange(NUM_CLASSES)
width = 0.25

plt.figure(figsize=(10, 5))
plt.bar(x - width, precision, width=width, label='Precision')
plt.bar(x, recall, width=width, label='Recall')
plt.bar(x + width, f1, width=width, label='F1-score')
plt.xticks(x, CLASS_NAMES, rotation=20)
plt.ylim(0, 1)
plt.ylabel('Score')
plt.title('Per-class test metrics')
plt.legend()
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

In [ ]:
# Save a concise metrics summary for the report table

model_size_mb = os.path.getsize(MODEL_PATH) / (1024 ** 2)

inference_start = time.time()
_ = evaluate_loader(model, test_loader, 'Test (timed run)')
inference_time = time.time() - inference_start

summary = {
    'best_epoch': best_epoch,
    'train_accuracy': float(train_acc_final),
    'validation_accuracy': float(val_acc_final),
    'test_accuracy': float(test_acc_final),
    'training_time_seconds': float(training_time),
    'test_inference_time_seconds': float(inference_time),
    'model_size_mb': float(model_size_mb),
}

summary_path = os.path.join(OUTPUT_DIR, 'cnn_summary_metrics.txt')
with open(summary_path, 'w') as f:
    for k, v in summary.items():
        f.write(f'{k}: {v}
')

print('Model size on disk: {:.2f} MB'.format(model_size_mb))
print('Timed inference on test set: {:.2f} seconds'.format(inference_time))
print('Saved summary to:', summary_path)

In [ ]:
# Qualitative results on random test samples
# The coursework asks for qualitative examples from the provided test set [file:1].

sample_indices = random.sample(range(len(test_samples)), min(12, len(test_samples)))
fig, axes = plt.subplots(3, 4, figsize=(12, 9))

for ax, idx in zip(axes.flat, sample_indices):
    img_path, true_label = test_samples[idx]
    pil_img = Image.open(img_path).convert('L')
    input_tensor = val_test_transform(pil_img).unsqueeze(0).to(device)

    with torch.no_grad():
        pred_label = model(input_tensor).argmax(dim=1).item()

    ax.imshow(pil_img, cmap='gray')
    title_color = 'green' if pred_label == true_label else 'red'
    ax.set_title(
        f'Pred: {CLASS_NAMES[pred_label]}\nTrue: {CLASS_NAMES[true_label]}',
        color=title_color,
        fontsize=9,
    )
    ax.axis('off')

plt.suptitle('Random test samples (green = correct, red = incorrect)')
plt.tight_layout()
plt.show()

In [ ]:
# Optional: helper for personal dataset / "in the wild" testing
# The coursework asks for an AgeDetection(path) function for the best model [file:1].
# This version assumes the folder contains face images already cropped to faces.


def AgeDetection(path, model_path=MODEL_PATH, n_images=4):
    image_paths = []
    for name in sorted(os.listdir(path)):
        ext = os.path.splitext(name)[1].lower()
        if ext in IMG_EXTS:
            image_paths.append(os.path.join(path, name))

    if len(image_paths) == 0:
        raise RuntimeError(f'No supported images found in {path}')

    chosen = random.sample(image_paths, min(n_images, len(image_paths)))

    model = AgeGroupCNN(num_classes=NUM_CLASSES).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()

    fig, axes = plt.subplots(1, len(chosen), figsize=(4 * len(chosen), 4))
    if len(chosen) == 1:
        axes = [axes]

    for ax, img_path in zip(axes, chosen):
        pil_img = Image.open(img_path).convert('L')
        input_tensor = val_test_transform(pil_img).unsqueeze(0).to(device)

        with torch.no_grad():
            pred = model(input_tensor).argmax(dim=1).item()

        ax.imshow(pil_img, cmap='gray')
        ax.set_title(f'{os.path.basename(img_path)}\nPred: {CLASS_NAMES[pred]}', fontsize=9)
        ax.axis('off')

    plt.suptitle('AgeDetection results')
    plt.tight_layout()
    plt.show()

In [ ]:
# Example call for personal dataset
# AgeDetection('/content/drive/MyDrive/CWFolder/Personal Dataset')